# Index check

This notebook looks at the index hack that I have used in the numerical mixing diagnostic.

## Background

I got stuck on the numerical mixing diagnostic for a very long time with it always producing output that was wrong.
Eventually, I saved the west and east values that I was accessing in `T_adx` and `uhtr` separately to compare to what was saved if the diagnostic is explicitly saved.
I found the indexing for `T_adx` lined up but the `uhtr` variable did not; it was shifted by one.
Once this was adjusted in the numerical mixing diagnostic, the output looked correct.

I have now written separate "diagnostics" to save the west and east u point values at each grid cell so that I have something to show to other people as a kind of mwe.
I save the standard diagnostic as well two other versions with `_wu` and `_eu` appended to indicate I have saved the west and east u points, respectively.
The saving is designed such that `_wu` corresponds to `1:end-1` and `_eu` corresponds to `2:end` in terms of indexing along the x direction.
This means I should have exact equality between

```julia
var[1:end-1, :, :, :] .== var_wu[:, :, :, :]
var[2:end, :, :, :] .== var_eu[:, :, :, :]
```

where `var` is a model variable saved on `u`-points and `var_wu` and`var_eu` are as described above.

In [3]:
tub_output_dir = "/g/data/e14/jb2381/anu-tub/outputs"
if pwd() != tub_output_dir 
    cd(tub_output_dir)
end
include("plotting.jl")

  Activating project at `/g/data/e14/jb2381/anu-tub/outputs`
[ Info: Plotting and analysis environment setup!
┌ Info: Experiments in the catalogue are:
│        - zstar-PPMH3
│        - hycom-PPMH3
│        Available data is:
│           - daily
│           - monthly
│           - monthlyz
│           - monthlyrho2
│           - static
└           - vertical coordinate


In [4]:
output = joinpath(pwd(), "anu-tub-nm-zstar-idxcheck", "output443", "ocean_hour.nc")

"/g/data/e14/jb2381/anu-tub/outputs/anu-tub-nm-zstar-idxcheck/output443/ocean_hour.nc"

## `T_adx`

In [22]:
ds = NCDataset(output, maskingvalue = NaN)

T_adx = ds["T_adx"][:, :, :, :]
find_nan = .!isnan.(T_adx)
T_adx .*= find_nan
T_adx_wu = ds["T_adx_wu"][:, :, :, :]
find_nan = .!isnan.(T_adx_wu)
T_adx_wu .*= find_nan
T_adx_eu = ds["T_adx_eu"][:, :, :, :]
find_nan = .!isnan.(T_adx_eu)
T_adx_eu .*= find_nan

close(ds)

closed Dataset

In [23]:
all(T_adx[1:end-1, :, :, :] .== T_adx_wu)

true

In [24]:
all(T_adx[2:end, :, :, :] .== T_adx_eu)

true

Can see with `T_adx`, the logic described above works as expected: the west upoints are `1:end-1` and the east upoints are `2:end`.

## `uhtr`

In [25]:
ds = NCDataset(output, maskingvalue = NaN)

uhtr = ds["uhtr"][:, :, :, :]
find_nan = .!isnan.(uhtr)
uhtr .*= find_nan
uhtr_wu = ds["uhtr_wu"][:, :, :, :]
find_nan = .!isnan.(uhtr_wu)
uhtr_wu .*= find_nan
uhtr_eu = ds["uhtr_eu"][:, :, :, :]
find_nan = .!isnan.(uhtr_eu)
uhtr_eu .*= find_nan

close(ds)

closed Dataset

In [26]:
all(uhtr[1:end-1, :, :, :] .== uhtr_wu)

false

In [27]:
all(uhtr[2:end, :, :, :] .== uhtr_eu)

false

Here, this logic does not hold and it is not just there is one value that does not match, none of them do.
However:

In [28]:
all(uhtr[1:end-1, :, :, :] .== uhtr_eu)

true

does hold.
This is what led to the hack in the numerical mixing code and what I need to get input on.